# 16.8 图注意力网络 GAT / Graph Attention Network

**中文**：GCN 聚合邻居时，权重是**固定的**——由度数决定($\hat A$ 里的 $1/\sqrt{d_i d_j}$)，和邻居"重不重要"无关。但直觉上，**不同邻居对你的价值不同**:一篇论文引用的 20 篇文献里，可能只有 3 篇真正决定它的主题。**GAT(Veličković et al., 2018)** 把 Transformer 的**注意力机制**搬到图上——让模型**自己学**每个邻居该分配多少权重。这是"图 + 注意力"的里程碑，也把我们 Part 12 学的自注意力和图连接了起来。
**English**: When GCN aggregates neighbors, the weights are **fixed** — determined by degree ($1/\sqrt{d_i d_j}$ in $\hat A$), regardless of how "important" a neighbor is. But intuitively **different neighbors matter differently**: among a paper's 20 citations, maybe only 3 truly determine its topic. **GAT (Veličković et al., 2018)** brings the **attention mechanism** from Transformers to graphs — letting the model **learn** how much weight each neighbor deserves. A milestone of "graph + attention," connecting the self-attention from Part 12 to graphs.

---

**中文**：GAT 的一层怎么算注意力(对节点 $i$ 及其邻居 $j$):
**English**: How one GAT layer computes attention (for node $i$ and neighbor $j$):

$$e_{ij}=\text{LeakyReLU}\big(\mathbf a^\top[W\mathbf h_i\,\|\,W\mathbf h_j]\big),\quad
\alpha_{ij}=\frac{\exp(e_{ij})}{\sum_{k\in N(i)}\exp(e_{ik})},\quad
\mathbf h_i'=\sigma\Big(\sum_{j\in N(i)}\alpha_{ij}\,W\mathbf h_j\Big)$$

**中文**：逐步解释:
**English**: Step by step:
1. **打分 $e_{ij}$**:先用共享权重 $W$ 变换 $i,j$ 的特征，拼接后与可学习向量 $\mathbf a$ 做内积、过 LeakyReLU——得到"$i$ 对 $j$ 的原始注意力分"。
   **Score $e_{ij}$**: transform $i,j$'s features with shared $W$, concatenate, dot with a learnable vector $\mathbf a$, apply LeakyReLU — the raw attention score of $i$ toward $j$.
2. **归一化 $\alpha_{ij}$**:对 $i$ 的**所有邻居**做 softmax，使权重和为 1(只在邻居间归一，不是全图)。
   **Normalize $\alpha_{ij}$**: softmax over $i$'s **neighbors** so weights sum to 1 (over neighbors only, not the whole graph).
3. **加权聚合**:用学到的 $\alpha$ 对邻居加权求和，过激活。
   **Weighted aggregation**: weighted sum of neighbors by the learned $\alpha$, then activation.

**中文**：和 Transformer 一样，GAT 用**多头注意力(multi-head)**:并行跑 $K$ 组独立注意力，中间层把结果**拼接**、输出层**平均**。多头能稳定训练、捕捉多种关系。
**English**: Like Transformers, GAT uses **multi-head attention**: run $K$ independent attention sets in parallel, **concatenate** them in hidden layers and **average** at the output. Multi-head stabilizes training and captures multiple relation types.

> 💡 **面试速查 / Interview cheat-sheet（★★★ GNN 必考）**
> **中文**：**GAT=在图上做注意力**:邻居权重不再由度数固定, 而是 $\alpha_{ij}$ **学出来**(softmax over 邻居)。对比:**GCN 权重固定/谱方法**, **GAT 权重自适应/空间方法**。**多头**稳定训练(concat 隐层, average 输出)。优点:①自适应权重(能忽略噪声邻居); ②**可解释**(看 $\alpha$ 知道模型关注谁); ③天然**归纳式**(和 GraphSAGE 一样只学函数, 不依赖固定图结构/谱)。诚实点:在同质图(Cora)上 GAT 与 GCN 准确率**相近**, 注意力的价值更多在异质/噪声图和可解释性。
> **English**: **GAT = attention on graphs**: neighbor weights are no longer fixed by degree but **learned** as $\alpha_{ij}$ (softmax over neighbors). Contrast: **GCN fixed weights / spectral**, **GAT adaptive weights / spatial**. **Multi-head** stabilizes training (concat in hidden, average at output). Pros: ① adaptive weights (can ignore noisy neighbors); ② **interpretable** ($\alpha$ shows who the model attends to); ③ naturally **inductive** (like GraphSAGE, learns a function, not tied to a fixed graph/spectrum). Honest note: on homophilous graphs (Cora), GAT and GCN reach **similar** accuracy; attention's value shows more on heterophilous/noisy graphs and in interpretability.


In [ ]:

# ============================================================
# 数据 Cora / Cora data
# ============================================================
import os, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
R=os.path.expanduser("~/.cache/dsfs_recsys/cora")
content=[l.split("\t") for l in open(os.path.join(R,"cora.content")).read().strip().split("\n")]
ids=[c[0] for c in content]; id2x={v:i for i,v in enumerate(ids)}; n=len(ids)
classes=sorted(set(c[-1] for c in content)); lab2y={c:i for i,c in enumerate(classes)}
y=torch.tensor([lab2y[c[-1]] for c in content])
X=torch.tensor(np.array([[int(x) for x in c[1:-1]] for c in content],dtype=np.float32))
X=X/X.sum(1,keepdim=True).clamp(min=1)
A=np.eye(n,dtype=np.float32)                                   # 含自环 / with self-loops
for line in open(os.path.join(R,"cora.cites")).read().strip().split("\n"):
    a,b=line.split("\t")
    if a in id2x and b in id2x: A[id2x[a],id2x[b]]=1; A[id2x[b],id2x[a]]=1
Amask=torch.tensor(A)
NEG=(-1e15)*(1-Amask)                                          # 非边处填 -inf, softmax 后为0 / mask non-edges
np.random.seed(0); tm=np.zeros(n,bool)
for c in range(len(classes)):
    idx=np.where(y.numpy()==c)[0]; np.random.shuffle(idx); tm[idx[:20]]=True
rest=np.where(~tm)[0]; np.random.shuffle(rest)
vm=np.zeros(n,bool); tem=np.zeros(n,bool); vm[rest[:500]]=True; tem[rest[500:1500]]=True
tm,vm,tem=torch.tensor(tm),torch.tensor(vm),torch.tensor(tem)
print(f"节点 {n}, 边(含自环) {int(Amask.sum())}, 类别 {len(classes)}, 标注 {tm.sum().item()}")


**中文**：从零实现 **多头 GAT 层**。用一个高效技巧算 $e_{ij}$:注意 $\mathbf a^\top[W\mathbf h_i\|W\mathbf h_j]=\mathbf a_1^\top W\mathbf h_i+\mathbf a_2^\top W\mathbf h_j$，所以先分别算每个节点的两个标量分数，再用广播相加得到 $n\times n$ 的注意力打分矩阵，用邻接掩码后按行 softmax。
**English**: Implement a **multi-head GAT layer** from scratch. Use an efficient trick for $e_{ij}$: note $\mathbf a^\top[W\mathbf h_i\|W\mathbf h_j]=\mathbf a_1^\top W\mathbf h_i+\mathbf a_2^\top W\mathbf h_j$, so compute two scalar scores per node, broadcast-add into an $n\times n$ score matrix, mask by adjacency, and softmax per row.


In [ ]:

# ============================================================
# 从零实现多头 GAT 层 / multi-head GAT layer from scratch
# ============================================================
class GATLayer(nn.Module):
    def __init__(s, fin, fout, heads, concat=True, drop=0.6):
        super().__init__(); s.heads=heads; s.concat=concat; s.drop=drop
        s.W =nn.Parameter(torch.empty(heads,fin,fout));  nn.init.xavier_uniform_(s.W)   # 每头一个变换 / per-head W
        s.a1=nn.Parameter(torch.empty(heads,fout));       nn.init.xavier_uniform_(s.a1) # 注意力向量分量 / a=[a1;a2]
        s.a2=nn.Parameter(torch.empty(heads,fout));       nn.init.xavier_uniform_(s.a2)
        s.last_alpha=None
    def forward(s, X):
        Wh=torch.einsum("nf,hfo->hno", X, s.W)                 # 变换特征 (heads,n,fout) / transform
        s1=(Wh*s.a1[:,None,:]).sum(-1)                         # a1·Wh_i  (heads,n)
        s2=(Wh*s.a2[:,None,:]).sum(-1)                         # a2·Wh_j  (heads,n)
        e=F.leaky_relu(s1[:,:,None]+s2[:,None,:], 0.2) + NEG[None]   # e_ij + 非边掩码 / masked scores
        alpha=F.softmax(e, dim=2)                              # 对邻居归一化 / softmax over neighbors
        s.last_alpha=alpha.detach()                            # 存下来做可视化 / keep for visualization
        alpha=F.dropout(alpha, s.drop, training=s.training)
        out=torch.einsum("hij,hjo->hio", alpha, Wh)            # 注意力加权聚合 / attention-weighted aggregate
        return out.permute(1,0,2).reshape(X.size(0),-1) if s.concat else out.mean(0)  # concat隐层/average输出

class GAT(nn.Module):
    def __init__(s, fin, h, fout, heads=8):
        super().__init__()
        s.l1=GATLayer(fin, h, heads, concat=True)              # 第1层:多头拼接 / hidden: concat heads
        s.l2=GATLayer(h*heads, fout, 1, concat=False)          # 第2层:单头输出 / output: single head
        s.dp=nn.Dropout(0.6)
    def forward(s, X):
        H=F.elu(s.l1(s.dp(X))); return s.l2(s.dp(H))

def train_gat(heads=8, epochs=120):
    torch.manual_seed(0); m=GAT(X.shape[1], 8, len(classes), heads)
    opt=torch.optim.Adam(m.parameters(), lr=0.005, weight_decay=5e-4); bv=0; bt=0; curve=[]
    for ep in range(epochs):
        m.train(); opt.zero_grad(); F.cross_entropy(m(X)[tm], y[tm]).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            pred=m(X).argmax(1); va=(pred[vm]==y[vm]).float().mean().item(); ta=(pred[tem]==y[tem]).float().mean().item()
            curve.append(ta)
            if va>bv: bv,bt=va,ta
    return bt, curve, m

t=time.time(); gat_acc, gat_curve, gat_model = train_gat(heads=8)
print(f"GAT (8头/heads) 测试准确率 / test accuracy: {gat_acc:.4f}  ({time.time()-t:.0f}s)")
print(f"(对比 16.6 的 GCN ≈ 0.77 / cf. GCN from 16.6 ≈ 0.77)")


**中文**：验证**多头的价值**:把注意力头数从 8 减到 1，看训练稳定性与精度如何变化。多头相当于"多个专家投票"，能降低单一注意力的方差。
**English**: Verify the **value of multiple heads**: reduce attention heads from 8 to 1 and see how training stability and accuracy change. Multiple heads act like "an ensemble of experts," reducing the variance of a single attention.


In [ ]:

# ============================================================
# 多头 vs 单头 / multi-head vs single-head
# ============================================================
t=time.time(); gat1_acc, gat1_curve, _ = train_gat(heads=1)
print(f"GAT 单头 (1 head)  test acc: {gat1_acc:.4f}")
print(f"GAT 多头 (8 heads) test acc: {gat_acc:.4f}   ({time.time()-t:.0f}s)")
print("→ 多头通常更稳更好一点 / multi-head is usually a bit more stable and better")


**中文**：GAT 的一大卖点是**可解释性**——原则上学到的 $\alpha_{ij}$ 能告诉你"判断节点 $i$ 时最看重哪个邻居"。下面挑一个节点、可视化它对各邻居的注意力，并和 GCN 的固定权重对比，**诚实地看看 GAT 在 Cora 上到底学出了怎样的注意力**(结果会很有启发)。
**English**: A key selling point of GAT is **interpretability** — in principle the learned $\alpha_{ij}$ tells you "which neighbor mattered most when judging node $i$." Below we pick a node, visualize its attention over neighbors versus GCN's fixed weights, and **honestly inspect what attention GAT actually learned on Cora** (the result is instructive).


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,5))
# ① 训练曲线:多头 vs 单头 / curves
ax[0].plot(gat_curve,label=f"8 heads ({gat_acc:.3f})",color="#4C72B0")
ax[0].plot(gat1_curve,label=f"1 head ({gat1_acc:.3f})",color="#DD8452")
ax[0].set_title("多头 vs 单头 / multi-head vs single"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("test acc"); ax[0].legend()
# ② 某节点的注意力权重 vs GCN 固定权重 / learned attention vs GCN fixed weights
gat_model.eval();
with torch.no_grad(): _=gat_model(X)
alpha=gat_model.l1.last_alpha.mean(0).numpy()               # 第一层多头平均注意力 / avg attention over heads
deg=Amask.sum(1).numpy()
# 选一个邻居较多的节点 / pick a node with several neighbors
node=int(np.argsort(deg)[len(deg)//2+40]); nbrs=np.where(Amask[node].numpy()>0)[0]
att=alpha[node,nbrs]                                        # GAT 学到的权重 / learned weights
gcn_w=1/np.sqrt(deg[node]*deg[nbrs]); gcn_w/=gcn_w.sum()    # GCN 的固定权重(归一化对比)/ GCN fixed weights
order=np.argsort(att)[::-1]
xpos=np.arange(len(nbrs))
ax[1].bar(xpos-0.2,att[order],0.4,label="GAT 学到 attention",color="#4C72B0")
ax[1].bar(xpos+0.2,gcn_w[order],0.4,label="GCN 固定权重",color="#C44E52")
ax[1].set_title(f"节点{node}对{len(nbrs)}个邻居的权重 / weights over neighbors"); ax[1].set_xlabel("邻居(按GAT注意力降序)"); ax[1].set_ylabel("weight"); ax[1].legend()
# ③ 注意力权重分布:GAT 学出了不均匀的关注 / distribution of attention entropy
ent=[]
for i in range(n):
    nb=np.where(Amask[i].numpy()>0)[0]; a=alpha[i,nb]; a=a/a.sum()
    ent.append(-(a*np.log(a+1e-12)).sum()/np.log(len(nb)) if len(nb)>1 else 1.0)  # 归一化熵 / normalized entropy
ax[2].hist(ent,bins=30,color="#55A868",edgecolor="white")
ax[2].axvline(np.mean(ent),color="red",ls="--",label=f"mean={np.mean(ent):.2f}")
ax[2].set_title("注意力熵(1=均匀, 0=集中) / attention entropy"); ax[2].set_xlabel("normalized entropy"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/g08_viz.png",dpi=80); plt.show()
print(f"注意力熵均值 / mean attention entropy = {np.mean(ent):.3f} (1=完全均匀/uniform)")
print("→ 在 Cora 上 GAT 学到的注意力几乎均匀 = 正是 GAT≈GCN 的原因 / near-uniform on Cora explains GAT≈GCN")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **在 Cora 上 GAT ≈ GCN**(都 ~0.77)。这很诚实也很重要:**注意力并不总是带来精度提升**。Cora 是强同质图——一个节点的邻居大多同类，"平均"和"加权平均"差别不大。GAT 的优势在**异质图/含噪声邻居**的场景(需要主动忽略无关邻居)、以及**可解释性**上更突出。别默认"有 attention 就更强"。
2. **多头更稳**:8 头通常比 1 头略好、且训练曲线更平稳——多头降低了单一注意力的方差(和 Transformer 一个道理)。
3. **诚实的意外——Cora 上注意力几乎均匀**:可视化算出的注意力熵均值≈1.0(1=完全均匀)，说明 GAT 在 Cora 上基本没有"挑邻居"，学出的权重和均匀聚合差不多。**这恰恰解释了为什么 GAT≈GCN**——同质图上邻居大多同类，注意力无用武之地，自然退化成"平均"。别把这当失败:它说明**注意力的价值取决于数据**。在噪声/异质图上，GAT 才会学出真正不均匀、可解释的注意力(主动忽略无关邻居)——那才是它的主场。

**English**:
1. **On Cora, GAT ≈ GCN** (both ~0.77). Honest and important: **attention does not always improve accuracy**. Cora is strongly homophilous — a node's neighbors are mostly same-class, so "average" and "weighted average" differ little. GAT's advantage stands out on **heterophilous / noisy-neighbor** settings (where you must actively ignore irrelevant neighbors) and in **interpretability**. Don't assume "attention = stronger."
2. **Multi-head is more stable**: 8 heads is usually a bit better than 1 and trains more smoothly — multi-head reduces a single attention's variance (same rationale as Transformers).
3. **An honest surprise — attention is near-uniform on Cora**: the measured mean attention entropy ≈ 1.0 (1 = fully uniform), meaning GAT barely "picks" neighbors here — its learned weights are close to plain averaging. **This is exactly why GAT ≈ GCN**: on a homophilous graph neighbors are mostly same-class, so attention has nothing to exploit and degenerates to "average." Don't read this as failure: it shows **attention's value is data-dependent**. On noisy/heterophilous graphs GAT learns genuinely non-uniform, interpretable attention (actively ignoring irrelevant neighbors) — that is its home turf.

> 💼 **实战视角 / Practical angle**
> **中文**:GAT 的地位——**注意力是 GNN 的通用增强件**, 很多现代 GNN(异质图 HGT、图 Transformer)都以它为基石。何时选 GAT:① 邻居质量参差、有噪声边(注意力能自动降权); ② 需要可解释(看模型关注哪些边); ③ 归纳式场景(不依赖图谱)。何时 GCN 就够:同质、干净、追求速度(GAT 因为算注意力更慢更吃内存)。**工程坑**:稠密注意力 $O(n^2)$ 不可扩展, 大图要用**稀疏/边级别注意力**(只对真实边算)。面试金句:*"GAT 把邻居权重从'度数决定'变成'学出来', 带来自适应和可解释; 但在同质图上未必比 GCN 准——注意力的价值场景是噪声/异质图。"*
> **English**: GAT's place — **attention is a general-purpose GNN enhancer**; many modern GNNs (heterogeneous HGT, graph Transformers) build on it. Choose GAT when: ① neighbor quality varies / noisy edges (attention down-weights them); ② you need interpretability (see which edges the model attends to); ③ inductive settings (no reliance on the spectrum). GCN suffices when: homophilous, clean, speed-critical (GAT is slower and more memory-hungry due to computing attention). **Engineering pitfall**: dense attention is $O(n^2)$ and unscalable — big graphs need **sparse / edge-level attention** (compute only over real edges). Interview line: *"GAT turns neighbor weights from 'degree-decided' into 'learned,' giving adaptivity and interpretability; but on homophilous graphs it needn't beat GCN — attention shines on noisy/heterophilous graphs."*

---
### 小结 / Summary
- **中文**:GAT=图上注意力, 邻居权重 $\alpha_{ij}$ 学出来(softmax over 邻居), 多头稳定训练。
- **English**: GAT = attention on graphs, neighbor weights $\alpha_{ij}$ learned (softmax over neighbors), multi-head for stability.
- **中文**:Cora 上 GAT≈GCN——注意力不总提精度; 其价值在噪声/异质图 + 可解释性。
- **English**: On Cora GAT≈GCN — attention doesn't always raise accuracy; its value is on noisy/heterophilous graphs + interpretability.
- **中文**:GCN 固定权重/谱、GAT 自适应权重/空间且天然归纳——三大 GNN(GCN/SAGE/GAT)各有侧重。
- **English**: GCN fixed-weight/spectral, GAT adaptive-weight/spatial and naturally inductive — the three GNNs (GCN/SAGE/GAT) each emphasize different things.
